# 2. WFS Propagation and Frame Preprocessing

Part 2 of the tutorial series (see `01_Dataset.ipynb` for `PhaseDataset`). This notebook covers the next two pipeline stages: the optical propagator (`AI4AO.PyramidWFS`, subclassing `AI4AO.TorchPropagator.WFS`), which turns a phase + pupil into a detector frame, and `AI4AO.FramePreprocess`, which crops that frame into individual pupil images and normalizes them — the actual input a reconstructor network consumes.

Still no DM and no reconstructor here; that's notebooks 3 and 4.

## Configuration, dataset and WFS

Same `wfs_params_exp.py` config as notebook 1. `PyramidWFS` builds its pyramid mask and reference intensity directly from `WFSParams` — there's no bench calibration behind this synthetic instrument, so there's no `LoadCalibration` call anywhere in this series.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, imshow, imshow_multiple

device = 'cuda'  # set to "cpu" if CUDA is not available

paramfile = 'wfs_params_exp.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']

AtmosParams['Scintillation'] = True

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)

wfs = PyramidWFS(WFSParams, device)
wfs.BuildMask()
wfs.BuildReferenceIntensity()

## Frame preprocessor

`FramePreprocess` will crop the 4 pyramid pupil images out of the raw detector frame and reference/normalize them. `ProcessReference` records the WFS's own flat-wavefront reference intensity now, to be subtracted from every subsequent frame.

In [ ]:
framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

## Drawing a phase screen and setting WFS noise

Each batch carries per-sample `"nphotons"`/`"ron"` values, drawn from the `Nphotons`/`RON` ranges in `WFSParams`. `wfs.SetPhotonsAndRON` must be called before propagation for the detector frame to include realistic Poisson + read noise (`WFSParams["useNoise"] = True` here).

In [ ]:
batch = dataset[0]
phaseGT = batch["phase"]
pupilGT = batch["pupil"]
wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

## Optical propagation

`wfs(phase, pupil)` propagates the input `phase` and `pupil` through the PWFS and returns the *raw* detector frame — a single 2D image containing all 4 pupil images together, normalized to `frame.sum() = 1`. To display the image we use the `imshow` function from AI4AO, which is made to handle both pytorch tensors and numpy arrays. It gives a simple syntax to show arrays of several images.

In [ ]:
with torch.no_grad():
    wfs_frame = wfs(phaseGT, pupilGT) # If the pupil is flat you can also just do wfs(phaseGT)

print(f'Detector frame shape = {wfs_frame.shape}')

plt.figure(figsize=(15, 5))
plt.subplot(131)
plt.title("Pupil amplitudes")
imshow(pupilGT, same_scale=True)
plt.subplot(132)
plt.title("Wavefront")
imshow(phaseGT, same_scale=True)
plt.subplot(133)
plt.title("WFS frame")
imshow(wfs_frame)
# plt.tight_layout()
plt.show()

## Watching the detector frame evolve with the turbulence

Stepping through `dataset[i]` for increasing `i` advances the same turbulence realization in time (wind-shifted per step — see `01_Dataset.ipynb`), so re-propagating each one animates how the pyramid frame reacts as the wavefront moves across the pupil. You can use the GetPSF method of the WFS class to get the current image of the source. You can select a specific `sampling` and field-of-view `fov` in units of \lambda/D 

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 50
phases, wfs_frames, psfs = [], [], []
for i in range(n_frames):
    batch = dataset[i]
    phase = batch["phase"]
    with torch.no_grad():
        wfs_frame = wfs(phase)
        psf = wfs.GetPSF(phase, sampling=4, fov=20)
    phases.append(phase)
    wfs_frames.append(wfs_frame)
    psfs.append(psf)

fig, axes = imshow_multiple([phases[0], wfs_frames[0], psfs[0]], same_scale=False, titles=["Wavefronts", "WFS frames", "PSF"])
# fig.set_dpi(80)

def update(i):
    imshow_multiple([phases[i], wfs_frames[i], psfs[i]], fig=fig, axes=axes, same_scale=False)
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())

## Frame preprocessing

`framePreprocessor.ProcessFrame` is what actually feeds a reconstructor network: it crops the 4 individual pupil images out of the raw detector frame (`FramePreprocess.GetPupils`), then reference-subtracts and normalizes them using the flat-wavefront reference recorded above. Compare the single raw frame on the left to the 4 separate, normalized pupil-image channels on the right. Note that they are not necessarily in the same order, but it does not matter as long as it is consistent. 
Here the imshow function takes the input tensor with size [BCWH] groups the images in the channel for display purposes.

In [ ]:
batch = dataset[0]
with torch.no_grad():
    wfs_frame = wfs.Propagator(batch["phase"])
    preprocessed_frames = framePreprocessor.ProcessFrame(wfs_frame)

print(f'Preprocessed frames shape = {preprocessed_frames.shape}')

plt.figure(figsize=(15, 8))
plt.subplot(121)
plt.title("WFS frames")
imshow(wfs_frame)
plt.subplot(122)
plt.title("Extracted pupils.")
imshow(preprocessed_frames)
plt.show()